# Productos más demandados según las compras del último mes

## Pregunta de negocio

¿Cuáles son los productos más demandados según las compras del último mes?

## Criterio de análisis

Se utiliza `bought_last_month` como indicador de demanda y se ordenan los productos de mayor a menor. En caso de empate, `rating_f` se usa únicamente como criterio secundario de ordenamiento. El análisis conserva tanto productos patrocinados como no patrocinados; la separación posterior permite contextualizar el primer lugar de cada grupo.

> **Alcance:** `bought_last_month` es la métrica disponible en el dataset. Por ello, las conclusiones describen las compras reportadas/estimadas en la fuente y no prueban que el patrocinio, el cupón o la calificación causen una mayor demanda.


In [2]:
from amazon_product_sales.utils.get_paths import build_paths
import pandas as pd

paths = build_paths()

df_cleaned = pd.read_csv(paths['processed'] / "clean_dataset.csv")

df_cleaned.columns

Index(['title', 'bought_last_month', 'rating_f', 'best_seller', 'sponsored',
       'has_coupon', 'coupon_pct', 'price_variant',
       'current_discounted_price_f', 'listed_price_f', 'number_reviews',
       'final_price'],
      dtype='object')

In [17]:
total_records = len(df_cleaned)
total_boughts = df_cleaned["bought_last_month"].sum()
sponsor_records = df_cleaned["sponsored"].sum()

# top_20 = df_cleaned.nlargest(20, "bought_last_month")[["title", "rating_f", "bought_last_month"]]

top_20 = (df_cleaned.sort_values(
    by=["bought_last_month", "rating_f"],
    ascending=[False, False],
)
.head(20)
)[["title", "rating_f", "bought_last_month", "has_coupon", "sponsored"]]

top_20_grouped = (
    top_20.groupby("title", as_index=False)
    .agg(
        rating_f=("rating_f", "mean"),
        bought_last_month=("bought_last_month", "sum"),
        has_coupon=("has_coupon", "max"),
        sponsored=("sponsored", "max")
    )
)

top_20_grouped

,title,rating_f,bought_last_month,has_coupon,sponsored
0,Amazon Basics 48-Pack AA Alkaline High-Perform...,4.7,100000.0,False,False
1,Amazon Basics AAA Alkaline High-Performance Ba...,4.7,100000.0,False,False
2,Amazon Basics Clear Thermal Laminating Plastic...,4.8,100000.0,False,False
3,"Amazon Basics Wood-Cased #2 Pencils, Pre-sharp...",4.8,100000.0,False,False
4,Duracell 2032 Lithium Battery. 4 Count Pack. C...,4.7,100000.0,False,False
5,"Energizer AA Batteries Alkaline Power, 32 Coun...",4.8,600000.0,False,True
6,"Energizer Alkaline Power AAA Batteries, 32 Cou...",4.8,630000.0,False,True
7,Texas Instruments TI-30XIIS Scientific Calcula...,4.7,100000.0,False,False
8,Texas Instruments TI-84 Plus CE Color Graphing...,4.6,100000.0,False,False


In [4]:
top_sponsored_true = top_20_grouped[top_20_grouped["sponsored"] == True]
top_sponsored_false = top_20_grouped[top_20_grouped["sponsored"] == False]

top_sponsored_false

,title,rating_f,bought_last_month,has_coupon,sponsored
0,Amazon Basics 48-Pack AA Alkaline High-Perform...,4.7,100000.0,False,False
1,Amazon Basics AAA Alkaline High-Performance Ba...,4.7,100000.0,False,False
2,Amazon Basics Clear Thermal Laminating Plastic...,4.8,100000.0,False,False
3,"Amazon Basics Wood-Cased #2 Pencils, Pre-sharp...",4.8,100000.0,False,False
4,Duracell 2032 Lithium Battery. 4 Count Pack. C...,4.7,100000.0,False,False
7,Texas Instruments TI-30XIIS Scientific Calcula...,4.7,100000.0,False,False
8,Texas Instruments TI-84 Plus CE Color Graphing...,4.6,100000.0,False,False


In [18]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=4,
    specs=[[{"type": "indicator"}] * 4]
)

# Tarjeta 1
fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_records,
        title={"text": "Total de registros"},
    ),
    row=1, col=1
)

# Tarjeta 2
fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_boughts,
        title={"text": "Total de compras"},
    ),
    row=1, col=2
)

# Tarjeta 3
fig.add_trace(
    go.Indicator(
        mode="number",
        value=sponsor_records,
        title={"text": "Articulos promocionados"},
    ),
    row=1, col=3
)

fig.update_layout(
    width=1200,
    height=250,
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()

In [45]:
top_title = top_sponsored_true.loc[top_sponsored_true["bought_last_month"].idxmax()]

top_title

title                Energizer Alkaline Power AAA Batteries, 32 Cou...
rating_f                                                           4.8
bought_last_month                                             630000.0
has_coupon                                                       False
sponsored                                                         True
Name: 6, dtype: object

In [57]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=4,
    specs=[[{"type": "indicator"}] * 4]
)

# Tarjeta 1
fig.add_trace(
    go.Indicator(
        mode="number",
        value=top_title['bought_last_month'],
        title={"text": "Compras último mes — patrocinado"},
    ),
    row=1, col=1
)

# Tarjeta 2
fig.add_trace(
    go.Indicator(
        mode="number",
        value=top_title['rating_f'],
        title={"text": "Calificación — patrocinado"},
    ),
    row=1, col=2
)

fig.update_layout(
    width=1200,
    height=250,
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()

In [56]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
top_title_no_sponsored = top_sponsored_false.loc[top_sponsored_false["bought_last_month"].idxmax()]

fig = make_subplots(
    rows=1,
    cols=4,
    specs=[[{"type": "indicator"}] * 4]
)

# Tarjeta 1
fig.add_trace(
    go.Indicator(
        mode="number",
        value=top_title_no_sponsored['bought_last_month'],
        title={"text": "Compras último mes — no patrocinado"},
    ),
    row=1, col=1
)

# Tarjeta 2
fig.add_trace(
    go.Indicator(
        mode="number",
        value=top_title_no_sponsored['rating_f'],
        title={"text": "Calificación — no patrocinado"},
    ),
    row=1, col=2
)

fig.update_layout(
    width=1200,
    height=250,
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()

In [50]:
import plotly.express as px

top_20_grouped["title"] = (
    top_20_grouped["title"]
    .str.split(r"[,.]")
    .str[0]
)

fig = px.bar(
    top_20_grouped,
    x="bought_last_month",
    y="title",
    orientation="h",
    color="rating_f",
    title="Ranking por título: suma de compras del último mes",
)

fig.update_layout(
    xaxis_title="Suma de compras durante el último mes",
    yaxis_title="Producto",
    coloraxis_colorbar_title="Calificación",
    yaxis={'categoryorder':'total ascending'}
)

fig.show()

## Respuesta y lectura de resultados

El cálculo actual identifica el título **Energizer Alkaline Power AAA Batteries, 32 Count** como el mayor valor agregado, con **630,000** compras, calificación de **4.8** y patrocinio activo. Sin embargo, ese total es la suma de **7 registros** que comparten el título; la compra máxima de una sola fila para ese producto es **90,000**.

En los productos no patrocinados, el máximo observado por título agregado es de **100,000** compras y varios artículos empatan en ese valor.
